# Can we predict `churn_next_month`?



Generated from `plan.yaml`. Every reported score trains on earlier values of `snapshot_month` than it tests on. See `design.md` for the reasons.

In [ ]:
import json

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, TimeSeriesSplit, train_test_split
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import OneHotEncoder

In [ ]:
df = pd.read_csv('data.csv')
df = df[df['churn_next_month'].notna()].reset_index(drop=True)
y = (df['churn_next_month'] == 1).astype(int)
features = ['seats', 'monthly_fee', 'tenure_months', 'usage_hours', 'support_tickets', 'plan']
X = df[features]

In [ ]:
SEED = 0
numeric = ['seats', 'monthly_fee', 'tenure_months', 'usage_hours', 'support_tickets']
categorical = ['plan']
prep = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numeric),
    ("cat", make_pipeline(SimpleImputer(strategy="most_frequent"),
                          OneHotEncoder(handle_unknown="infrequent_if_exist", max_categories=20,
                                        sparse_output=False)), categorical),
])
models = {
    "baseline": Pipeline([("prep", clone(prep)), ("model", DummyClassifier(strategy="prior"))]),
    "candidate": Pipeline([("prep", clone(prep)),
                           ("model", HistGradientBoostingClassifier(early_stopping=False,
                                                                    random_state=SEED))]),
}

In [ ]:
t = pd.to_datetime(df['snapshot_month'], errors="coerce", format="mixed")
periods = np.sort(t.dropna().unique())
cutoff = periods[int(len(periods) * 0.8)]
train = (t < cutoff).to_numpy()
test = (t >= cutoff).to_numpy()
print(f"train {t[train].min()} to {t[train].max()}, test {t[test].min()} to {t[test].max()}")
print(f"undated rows left out: {int(t.isna().sum())}")

In [ ]:
train_periods = np.sort(t[train].unique())
cv = TimeSeriesSplit(n_splits=5)
folds = [(np.flatnonzero(t.isin(train_periods[a]).to_numpy()),
          np.flatnonzero(t.isin(train_periods[b]).to_numpy()))
         for a, b in cv.split(train_periods)]

In [ ]:
def scores(y_true, p):
    return {
        "roc_auc": float(roc_auc_score(y_true, p)),
        "average_precision": float(average_precision_score(y_true, p)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, p >= 0.5)),
    }


results = {"cv": {}, "holdout": {}, "skipped_folds": 0}
usable = [(a, b) for a, b in folds if y.iloc[b].nunique() == 2]
results["skipped_folds"] = len(folds) - len(usable)
for name, model in models.items():
    results["cv"][name] = []
    for a, b in usable:
        model.fit(X.iloc[a], y.iloc[a])
        results["cv"][name].append(scores(y.iloc[b], model.predict_proba(X.iloc[b])[:, 1]))
    model.fit(X[train], y[train])
    results["holdout"][name] = scores(y[test], model.predict_proba(X[test])[:, 1])
print("WALDEAN_METRICS " + json.dumps(results, sort_keys=True))
pd.DataFrame(results["holdout"]).T

Both models are scored once on the test rows. Neither is chosen by that score. The CV scores come from the training rows only.